# Draw IO sandbox
Requires N2G and ttp installed in /lib folder.
These libraries are not available from official Anaconda or pip distribution channels.

Check out `igraph <https://igraph.org/python/doc/tutorial/tutorial.html#layout-algorithms>`

In [ ]:
source = '/Users/bue/dev/borr/2021-07-14 BORR IM/DB/DC-IM.json'

In [ ]:
import os
import sys
import json
from lxml import etree

In [ ]:
with open(source, 'r') as src:
    ssot = json.load(src)
assert len(list(ssot['diagrams'])) > 0, f"Cannot find any diagram in {ssot['model']}"

In [ ]:
diagram = ssot['diagrams']['DIAG1807'] # Product
#diagram = ssot['diagrams']['DIAG1806'] # Preisbildung
diagram = ssot['diagrams']['DIAG1805'] # Arbeitsplanung
diagram = ssot['diagrams']['DIAG1812'] # Vertrag
diagram = ssot['diagrams']['DIAG1809'] # Wissen

lang = ssot['model']['language']

In [ ]:
def entity_name(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    sn = enti['name'][lang]
    return sn

In [ ]:
def entity_description(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    descr = enti['descr'][lang]
    if isinstance(descr, str):
        return descr
    else:
        #print(f"Entity {key} has no description!\n{enti}")
        return None

In [ ]:
def entity_synonyms(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    synlist = list(enti['synonyms'].values())
    if len(synlist) > 0:
        result = []
        for synonym in synlist: 
            word = synonym.get(lang)
            if isinstance(word, str) and len(word) > 0:
                result.append(word)
        return ', '.join(result)
    return None

In [ ]:
entity_name('ENTI387')

In [ ]:
diagram_xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<mxfile host="Electron" modified="2021-07-20T12:02:15.557Z" agent="curl/7.1" etag="25mQkM6mx7LJW4tu3GDx" version="14.6.13" type="device">
  <diagram id="-IuDeWdp_pBGzQphX35I" name="{diagram['name']}">
    <mxGraphModel dx="{diagram['width']}" dy="{diagram['height']}" pageWidth="{diagram['width']}" pageHeight="{diagram['height']}" grid="1" gridSize="10" guides="1" tooltips="1" connect="1" arrows="1" fold="1" page="1" pageScale="1" math="0" shadow="0">
      <root>
        <mxCell id="0" />
        <mxCell id="1" parent="0" />
      </root>
    </mxGraphModel>
  </diagram>
</mxfile>"""

In [ ]:
from io import BytesIO

parser = etree.XMLParser(remove_blank_text=True)
dom = etree.parse(BytesIO(diagram_xml.encode('utf-8')), parser)

#dom = etree.fromstring(diagram_xml.encode('utf-8'), remove_blank_text=True)

In [ ]:
root = dom.find('.//root')

In [ ]:
len(root.getchildren())

In [ ]:
import spectra
black = spectra.html('#000000')

In [ ]:
entity_xml = """<mxcell id="{id}" value="{name}" sytle="{style}" parent="1" vertex="1">"""
entity_style = "rounded=1;whiteSpace=wrap;html=1;align=center;verticalAlign=top;"

In [ ]:
visible = set( element['element'] for element in diagram['elements']['entity'])
len(visible), list(visible)[:2]

## Add entities

In [ ]:
for element in diagram['elements']['entity']:
    enti_key = element['element']
    enti = ssot['entities'][enti_key]
    
    uo = etree.Element('UserObject')
    uo.set('id', enti_key)
    uo.set('label', entity_name(enti_key))
    uo.set('link', 'ssot:' + enti_key)
    descr = entity_description(enti_key)
    if descr:
        uo.set('Beschreibung', descr)
    
    synonyms = entity_synonyms(enti_key)
    if synonyms:
        uo.set('Synonyme', synonyms)
    
    fillcolor = element['ui']['color']
    fill = spectra.html('#' + fillcolor)
    
    supertypes = enti.get('supertypes+', [])
    if len(supertypes) > 0 and len(visible.intersection(supertypes)) > 0:
        #print(f"Brightening up {enti_key}")
        fill = fill.brighten(amount=5)
    
    cell = etree.Element("mxCell", id=enti_key + '-cell', style=entity_style + f"fillColor={fill.hexcode};", parent='1', vertex='1')
    box = etree.Element("mxGeometry", x=str(element['pos_x']), y=str(element['pos_y']), 
                            width=str(element['ui']['width']), height=str(element['ui']['height']))
    box.set('as', 'geometry')
    
    cell.append(box)
    uo.append(cell)
    root.append(uo)

## Add relations


```
<mxCell id="_I7dXOIRE_6SKKpCf0Ln-2" value=""style="endArrow=classic;startArrow=classic;html=1;exitX=1;exitY=0.5;exitDx=0;exitDy=0;jumpStyle=none;edgeStyle=elbowEdgeStyle;" edge="1" parent="1" source="ENTI372" target="ENTI421">
    
            <mxGeometry width="50" height="50" relative="1" as="geometry">
                <mxPoint x="1740" y="1310" as="sourcePoint" />?
                <mxPoint x="1894" y="1070" as="targetPoint" />?
            </mxGeometry>
</mxCell>

<mxCell id="_I7dXOIRE_6SKKpCf0Ln-3" value="Connector 1234" style="edgeLabel;html=1;align=center;verticalAlign=middle;resizable=0;points=[];" vertex="1" connectable="0" parent="_I7dXOIRE_6SKKpCf0Ln-2">
            <mxGeometry x="0.4318" y="-2" relative="1" as="geometry">
                <mxPoint as="offset" />
            </mxGeometry>
</mxCell>
```

```
<mxCell id="_I7dXOIRE_6SKKpCf0Ln-1" value="" style="endArrow=none;html=1;" edge="1" parent="1">
      <mxGeometry width="50" height="50" relative="1" as="geometry">
        <mxPoint x="790" y="1990" as="sourcePoint" />
        <mxPoint x="1150" y="2030" as="targetPoint" />
        <Array as="points">
          <mxPoint x="930" y="1780" />
          <mxPoint x="1100" y="1880" />
        </Array>
      </mxGeometry>
</mxCell>

<mxCell id="_I7dXOIRE_6SKKpCf0Ln-4" value="3 Segement Line bue3" style="edgeLabel;html=1;align=center;verticalAlign=middle;resizable=0;points=[];" vertex="1" connectable="0" parent="_I7dXOIRE_6SKKpCf0Ln-1">
      <mxGeometry x="-0.4516" y="-1" relative="1" as="geometry">
        <mxPoint x="-1" as="offset" />
      </mxGeometry>
</mxCell>
```

In [ ]:
print(f"Adding {len(diagram['relationships'])} relations")

In [ ]:
style="html=1;exitX=1;exitY=0.5;exitDx=0;exitDy=0;jumpStyle=none;edgeStyle=elbowEdgeStyle;"

In [ ]:
def get_connector(cardinality: str, mandatory: bool = False) -> str:
    if 'M' == cardinality:
        return 'ERoneToMany' if mandatory else 'ERmany' 
    elif '1' == cardinality:
        return 'ERzeroToOne' if mandatory else 'ERone'
    return 'none'

In [ ]:
def add_start_label(key: str, relation_ssot: dict, root):
    # lablels x value in mxGeometry specifies the position along the WHOLE connector: -1=start, 0=halfway between start end end, 1=end
    labeltext = relation_ssot['from-to'].get('assoc').get(lang)
    if labeltext is not None and len(labeltext) > 0:
        start_label = etree.Element('mxCell', value=labeltext, parent=key, vertex="1", connectable="0",
                                    style="edgeLabel;html=1;align=center;verticalAlign=middle;resizable=0;points=[];")
        start_label.set('id', key + '-sl')
        start_box = etree.Element('mxGeometry', relative="1",
                                  #x=str(relation['starttext_x']), y=str(relation['starttext_y']),
                                  x="-0.85",
                                  #width=str(relation['starttext_width']), height=str(relation['starttext_height']))
                                 )
        start_box.set('as', 'geometry')
        start_point = etree.Element('mxPoint')
        start_point.set('as', 'offset')                  
        start_box.append(start_point)
        start_label.append(start_box)
        root.append(start_label)


def add_end_label(key:str, relation_ssot: dict, root):
    labeltext = relation_ssot['to-from'].get('assoc').get(lang)
    if labeltext is not None and len(labeltext) > 0:
        end_label = etree.Element('mxCell', value=labeltext, parent=key, vertex="1", connectable="0",
                                    style="edgeLabel;html=1;align=center;verticalAlign=middle;resizable=0;points=[];")
        end_label.set('id', key + '-el')
        end_box = etree.Element('mxGeometry', relative="1",
                                  #x=str(relation['endtext_x']), y=str(relation['endtext_y']),
                                  x="0.85",
                                  #width=str(relation['endtext_width']), height=str(relation['endtext_height'])
                               )
        end_box.set('as', 'geometry')
        end_point = etree.Element('mxPoint')
        end_point.set('as', 'offset')
        end_box.append(end_point)
        end_label.append(end_box)
        root.append(end_label)

In [ ]:
for key, relation in diagram['relationships'].items():
    segments = relation['linesegments']
    assert len(segments) > 2, f"Expecting at least 2 points"
    start = segments[0]
    end = segments[-1]
    elbows = segments[1:-1]
    
    connector = etree.Element('mxCell', edge="1", parent="1", width="50", height="50")
    connector.set('id', key)
    geo = etree.Element('mxGeometry', width='50', height='50', relative='1')
    geo.set('as', 'geometry')
    
    start_node = etree.Element('mxPoint', x=str(start['x']), y=str(start['y']))
    start_node.set('as', 'sourcePoint')
    geo.append(start_node)
    
    end_node = etree.Element('mxPoint', x=str(end['x']), y=str(end['y']))
    end_node.set('as', 'targetPoint')
    geo.append(end_node)
    
    if len(elbows) > 0:
        elbows = etree.Element('Array')
        elbows.set('as', 'points')
        for linesegment in elbows:
            segment = relation['linesegments'][linesegment]
            elbow = etree.Element('mxPoint', x=str(segment['x']), y=str(segment('y')))
            elbows.append(elbow)
            
        geo.append(elbows)
    
    # Set end's
    relation_ssot = ssot['relations'][key]
    start_type = get_connector(relation['start_connector'], relation_ssot['from-to'].get('mandatory') )
    end_type = get_connector(relation['end_connector'])
    connector.set('style', style + f'startArrow={start_type};endArrow={end_type};')

    connector.append(geo)
    root.append(connector)
    
    # add_start_label(key, relation_ssot, root)
    # add_end_label(key, relation_ssot, root)
    labeltext = relation_ssot['from-to'].get('assoc').get(lang)
    if labeltext is not None and len(labeltext) > 0:
        start_label = etree.Element('mxCell', value=labeltext, parent="1", vertex="1",
                                    style="text;html=1;strokeColor=none;fillColor=none;align=center;verticalAlign=middle;whiteSpace=wrap;rounded=0;labelBackgroundColor=#FFFFFF;")
        start_label.set('id', key + '-sl')

        #<mxGeometry x="80" y="20" width="160" height="20" as="geometry" />
        start_label_box = etree.Element('mxGeometry', x=str(int(relation['starttext_x'])+2), y=str(int(relation['starttext_y'])+2),
                                       width=str(int(relation['starttext_width'])-4), height=str(int(relation['starttext_height']-4)))
        start_label_box.set('as', 'geometry')
        start_label.append(start_label_box)
        root.append(start_label)
                
    labeltext = relation_ssot['to-from'].get('assoc').get(lang)
    #if labeltext is not None and len(labeltext) > 0:
    

In [ ]:
graph = dom.find('.//mxGraphModel')

In [ ]:
destination_name = f"graph-{diagram['name']}.drawio"
with open(destination_name, 'wb') as out:
    out.write(etree.tostring(dom, encoding='utf-8'))
print(f"Wrote {destination_name}")

In [ ]:
import IPython
IPython.display.Code(etree.tostring(graph, pretty_print=True).decode('UTF-8'))